In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
DATA_PATH = "../data/raw/LC_loans_granting_model_dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

In [ ]:
print("DATASET DIMENSIONS")
print(df.shape)

In [ ]:
print("\nVARIABLES AND TYPES")
print(df.dtypes.to_string())

In [ ]:
print("\nMISSING VALUES")
print(df.isna().sum().to_string())

In [ ]:
print("\nTARGET DISTRIBUTION")
print(df["Default"].value_counts(dropna=False))
print(df["Default"].value_counts(normalize=True, dropna=False))

In [ ]:
print("\nDATA COVERAGE")
dates = pd.to_datetime(df["issue_d"], format="%b-%Y")
print("Earliest:", dates.min())
print("Latest:", dates.max())

In [ ]:
print("\nLOANS BY YEAR")
print(dates.dt.year.value_counts().sort_index())

In [ ]:
print("\nDEFAULT RATE BY YEAR")
annual_default = (
    df.assign(year=dates.dt.year)
    .groupby("year")["Default"]
    .agg(["count","mean"])
)

annual_default["mean"] *= 100
annual_default = annual_default.rename(columns={"count":"loans", "mean":"default_rate_pct"})

print(annual_default.round(2).to_string())

In [ ]:
print("\nFEATURE MEANS BY YEAR")

annual_features = (
    df.assign(year=dates.dt.year)
    .groupby("year")[["fico_n","dti_n","loan_amnt"]]
    .mean()
)

print(annual_features.round(2).to_string())

In [ ]:
fico_bins = [300, 600, 650, 700, 750, 800, 850]

fico_data = df.copy()
fico_data["fico_group"] = pd.cut(
    fico_data["fico_n"],
    bins=fico_bins,
    include_lowest=True
)

fico_default = (
    fico_data.groupby("fico_group", observed=True)["Default"]
    .agg(["mean", "count"])
)

fico_default["mean"] *= 100

display(fico_default)

fico_default["mean"].plot(
    kind="bar",
    figsize=(9, 4),
    ylabel="Default rate (%)",
    xlabel="FICO range",
    title="Observed default rate by FICO range",
    rot=45
)

plt.tight_layout()
plt.show()

In [ ]:
# Define origination periods
periods = pd.cut(
    dates.dt.year,
    bins=[2006, 2012, 2015, 2017, 2018],
    labels=["2007–2012", "2013–2015", "2016–2017", "2018"]
)

# Calculate default rates by FICO range and period
comparison = (
    df.assign(period=periods, fico_group=fico_data["fico_group"])
    .groupby(["fico_group", "period"], observed=True)["Default"]
    .agg(["mean", "count"])
)

comparison["mean"] *= 100

display(comparison)

In [ ]:
ax = (
    comparison["mean"]
    .unstack("period")
    .plot(
        kind="bar",
        figsize=(11, 5),
        ylabel="Default rate (%)",
        xlabel="FICO range",
        title="Default rates by FICO range and origination period",
        rot=45,
    )
)

ax.legend(title="Origination period")
plt.tight_layout()
plt.show()

In [ ]:
# Define FICO categories
fico_bins = [300, 600, 650, 700, 750, 800, 850]

df["fico_group"] = pd.cut(df["fico_n"], bins=fico_bins, include_lowest=True)

# Define DTI categories
dti_bins = [0, 10, 20, 30, 40, float("inf")]
dti_labels = ["0–10", "10–20", "20–30", "30–40", "40+"]

df["dti_group"] = pd.cut(df["dti_n"], bins=dti_bins, labels=dti_labels, include_lowest=True)

# Calculate default rates for combinations of FICO and DTI
joint_risk = (df.groupby(["fico_group", "dti_group"], observed=True)["Default"].agg(["mean", "count"]))

joint_risk["mean"] *= 100

display(joint_risk)

In [ ]:
risk_matrix = joint_risk["mean"].unstack("dti_group")
count_matrix = joint_risk["count"].unstack("dti_group")

fig, ax = plt.subplots(figsize=(10, 5))

im = ax.imshow(risk_matrix, aspect="auto", cmap="YlOrRd")

ax.set_xticks(range(len(risk_matrix.columns)))
ax.set_xticklabels(risk_matrix.columns)

ax.set_yticks(range(len(risk_matrix.index)))
ax.set_yticklabels([str(x) for x in risk_matrix.index])

ax.set_xlabel("Debt-to-income ratio")
ax.set_ylabel("FICO range")
ax.set_title("Observed default rate by FICO and DTI")

fig.colorbar(im, ax=ax, label="Default rate (%)")

plt.tight_layout()
plt.show()

In [ ]:
numeric_features = ["revenue","dti_n","loan_amnt","fico_n","experience_c"]

# Summary statistics, including extreme quantiles
display(df[numeric_features].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T)

# Check values outside expected ranges
print("Negative income:", (df["revenue"] < 0).sum())
print("Negative DTI:", (df["dti_n"] < 0).sum())
print("Non-positive loan amounts:", (df["loan_amnt"] <= 0).sum())
print("FICO outside 300–850:", (~df["fico_n"].between(300, 850)).sum())

In [ ]:
# Investigate extreme DTI values
print("DTI >= 40:", (df["dti_n"] >= 40).sum())
print("DTI >= 100:", (df["dti_n"] >= 100).sum())
print("DTI == 999:", (df["dti_n"] == 999).sum())

print("\nLargest DTI values:")
display(df["dti_n"].value_counts().sort_index().tail(15))

In [ ]:
# Investigate experience_c
print("\nExperience variable distribution:")
display(df["experience_c"].value_counts(dropna=False))

print("\nDefault rate by experience_c:")
display(df.groupby("experience_c")["Default"].agg(["count", "mean"]))

In [ ]:
categorical_features = ["emp_length","purpose","home_ownership_n","addr_state"]

for feature in categorical_features:
    print(f"\n{feature}")
    print("Unique categories:", df[feature].nunique(dropna=False))
    display(df[feature].value_counts(dropna=False))

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from underwriting.data import temporal_split

splits = temporal_split(df)

for name, subset in splits.items():
    print(f"{name}: {len(subset):,} loans")

In [ ]:
Path.cwd().resolve().parent

In [ ]:
sys.path